# ACDC — Exploração inicial
Conhecer os dados e identificar problemas antes da limpeza.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 40)

In [ ]:
df_raw = pd.read_csv("../data/raw/acdc_dataset.csv", sep=";")
df = df_raw.copy()
df.shape

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df.describe().round(2)

## Missing values e duplicados

In [ ]:
missing = df.isna().sum()
missing[missing > 0].sort_values(ascending=False)

In [ ]:
pd.Series({
    "Linhas duplicadas": df.duplicated().sum(),
    "IDs repetidos": df["member_id"].duplicated().sum()
})

## Categorias

In [ ]:
df[["gender", "country", "industry", "plan_tier"]].nunique()

In [ ]:
df["plan_tier"].value_counts(dropna=False)

In [ ]:
df["gender"].value_counts(dropna=False)

In [ ]:
df["country"].value_counts(dropna=False).head(15)

In [ ]:
df["industry"].value_counts(dropna=False).head(15)

## Valores a rever
Os limites de idade abaixo são exploratórios, não regras de remoção.

In [ ]:
numeric_columns = df.select_dtypes(include="number").columns
negatives = (df[numeric_columns] < 0).sum()
negatives[negatives > 0]

In [ ]:
pd.Series({
    "Idade fora de 18–100": ((df["age"] < 18) | (df["age"] > 100)).sum(),
    "Plano com zero dias por semana": (df["plan_days_per_week"] == 0).sum(),
    "Plano com mais de sete dias por semana": (df["plan_days_per_week"] > 7).sum(),
    "Visitas permitidas iguais a zero": (df["entitled_visits_last_period"] == 0).sum(),
    "Visitas reais acima das permitidas": (df["actual_visits_last_period"] > df["entitled_visits_last_period"]).sum()
})

## Formato da receita
Uma falha de conversão não significa que o valor original esteja em falta.

In [ ]:
revenue_numeric = pd.to_numeric(df["lifetime_value"], errors="coerce")
conversion_issues = df["lifetime_value"].notna() & revenue_numeric.isna()

print("Valores não vazios que precisam de conversão:", conversion_issues.sum())
print("Receitas negativas já legíveis:", (revenue_numeric < 0).sum())
df.loc[conversion_issues, ["lifetime_value"]].head(8)

## Datas
Verificar os dois formatos presentes numa tabela auxiliar.

In [ ]:
dates = pd.DataFrame(index=df.index)

for column in ["membership_start", "membership_end", "last_badge_swipe"]:
    iso_dates = pd.to_datetime(df[column], format="%Y-%m-%d", errors="coerce")
    other_dates = pd.to_datetime(df[column], format="%d/%m/%Y", errors="coerce")
    dates[column] = iso_dates.fillna(other_dates)

dates.isna().sum()

In [ ]:
dates.agg(["min", "max"])

In [ ]:
end_1970 = dates["membership_end"] == pd.Timestamp("1970-01-01")

pd.Series({
    "Fim em 1970-01-01": end_1970.sum(),
    "Fim antes do início (sem 1970)": ((dates["membership_end"] < dates["membership_start"]) & ~end_1970).sum(),
    "Última entrada antes do início": (dates["last_badge_swipe"] < dates["membership_start"]).sum(),
    "Última entrada após o fim (sem 1970)": ((dates["last_badge_swipe"] > dates["membership_end"]) & ~end_1970).sum()
})

## Primeiros gráficos

In [ ]:
df[["age", "total_visits", "avg_weekly_visits"]].hist(
    bins=30, figsize=(12, 3), layout=(1, 3)
)
plt.tight_layout()
plt.show()

In [ ]:
df["plan_tier"].value_counts().sort_values().plot(kind="barh", figsize=(8, 5))
plt.xlabel("Número de membros")
plt.ylabel("Plano (valores originais)")
plt.tight_layout()
plt.show()

## Próximo passo
Rever duplicados, categorias, missing values, valores suspeitos e datas. Confirmar o significado de `1970-01-01`. Ainda não foram removidas linhas nem imputados valores.